In [1]:
import os, time, requests, pandas as pd
from datetime import datetime, timedelta
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))
KEY = os.getenv("SOLAR_API_KEY")
URL = "https://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getVilageFcst"
print("키 로드:", bool(KEY))

키 로드: True


In [2]:
def call(bd, bt, nx=53, ny=38):
    p = dict(serviceKey=KEY, pageNo="1", numOfRows="1000", dataType="JSON",
             base_date=bd, base_time=bt, nx=str(nx), ny=str(ny))
    try:
        j = requests.get(URL, params=p, timeout=30).json()
        h = j["response"]["header"]
        items = j["response"].get("body", {}).get("items", {})
        its = items.get("item", []) if isinstance(items, dict) else []
        first = its[0]["fcstDate"] + its[0]["fcstTime"] if its else None
        return h["resultCode"], h["resultMsg"], len(its), first
    except Exception as e:
        return "ERR", str(e)[:60], 0, None

rows = []
for d in [0, 1, 2, 3, 4, 5, 7, 10, 30, 90, 365]:
    bd = (datetime.now() - timedelta(days=d)).strftime("%Y%m%d")
    for bt in ["0200", "1400", "2300"]:
        code, msg, n, first = call(bd, bt)
        rows.append(dict(일전=d, base_date=bd, base_time=bt, code=code, msg=msg, 건수=n, 첫예보=first))
        time.sleep(0.3)
res = pd.DataFrame(rows)
res

,일전,base_date,base_time,code,msg,건수,첫예보
0,0,20261027,0200,00,NORMAL_SERVICE,944,202610270300
1,0,20261027,1400,00,NORMAL_SERVICE,798,202610271500
2,0,20261027,2300,10,현재 이후의 base_time은 호출할 수 없습니다.,0,None
3,1,20261026,0200,00,NORMAL_SERVICE,944,202610260300
4,1,20261026,1400,00,NORMAL_SERVICE,798,202610261500
5,1,20261026,2300,00,NORMAL_SERVICE,980,202610270000
6,2,20261025,0200,00,NORMAL_SERVICE,944,202610250300
7,2,20261025,1400,00,NORMAL_SERVICE,798,202610251500
8,2,20261025,2300,00,NORMAL_SERVICE,980,202610260000
9,3,20261024,0200,10,최근 3일 간의 자료만 제공합니다.,0,None


In [3]:
ok = res[res["건수"] > 0]
print("가장 오래된 base_date:", ok["base_date"].min() if len(ok) else None)

가장 오래된 base_date: 20261024
